# Single-unit heatmaps from max-rayleigh tuned IDs

This notebook:
1. Loads tuned IDs from:
   - `tuned_ids_A_preflip_AND_maxrayleighA.csv`
   - `tuned_ids_B_postflip_AND_maxrayleighB.csv`
2. Finds each session's `processed_data` folder.
3. Plots one 3-panel heatmap per tuned cell (`shelter_only`, `barrier_pre_flip`, `barrier_post_flip`).


In [ ]:
from pathlib import Path
import re
import gc

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe

from IPython.display import Image, display


In [ ]:
# -----------------------
# Paths / settings
# -----------------------
SAVE_ROOT = Path(r"/ceph/branco/Jasmine_Laurence/rayleigh_analysis/Top2_TunED")
EXPERIMENTAL_DATA_ROOT = Path(r"/ceph/branco/Jasmine_Laurence/Experimental_Data")
OUT_DIR = SAVE_ROOT / "single_unit_heatmaps_maxrayleigh_tuned_ids"

A_IDS_CSV = SAVE_ROOT / "tuned_ids_A_preflip_AND_maxrayleighA.csv"
B_IDS_CSV = SAVE_ROOT / "tuned_ids_B_postflip_AND_maxrayleighB.csv"

FPS = 40
NBINS = 30
COLORMAP = "magma"
DPI = 300

CONDITIONS = [
    ("shelter", "shelter_only"),
    ("barrier_pre_flip", "barrier_pre_flip"),
    ("barrier_post_flip", "barrier_post_flip"),
]

# Optional limit while testing; set to None for all cells



# Optional limit while testing; set to None for all cells
MAX_PLOTS = 50

# Start with one mouse family to avoid kernel pressure (e.g., ["JAL4"])
SESSION_PREFIX_FILTER = ["JAL4"]

# Notebook display controls (saved PNGs are always written)
SHOW_INLINE = False
MAX_IMAGE_PREVIEWS = 5


In [ ]:
# -----------------------
# Helpers
# -----------------------
MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}


def _short_month(mm: int) -> str:
    arr = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    return arr[mm - 1]


def _mouse_num(name: str):
    m = re.search(r"JAL0*(\d+)", name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.match(r"0*(\d+)_", name)
    if m:
        return int(m.group(1))
    return None


def _flip_num(name: str):
    m = re.search(r"flip[_ ]?(\d+)", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _day_month_from_target(name: str):
    m = re.search(r"(\d{1,2})(?:st|nd|rd|th)?([A-Za-z]{3,9})", name, flags=re.IGNORECASE)
    if not m:
        return None
    dd = int(m.group(1))
    mon_raw = m.group(2).lower()
    if mon_raw not in MONTHS:
        return None
    mm = MONTHS[mon_raw]
    return (dd, mm)


def _day_month_from_discovered(name: str):
    m = re.search(r"(20\d{2})_(\d{2})_(\d{2})", name)
    if m:
        mm = int(m.group(2))
        dd = int(m.group(3))
        return (dd, mm)
    return _day_month_from_target(name)


def _target_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_target(name)
    keys = []
    if mouse is None:
        return keys
    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def _discovered_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_discovered(name)
    keys = []
    if mouse is None:
        return keys
    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def resolve_session_names(target_sessions, discovered_session_names):
    key_to_discovered = {}
    for ds in discovered_session_names:
        for k in _discovered_alias_keys(ds):
            key_to_discovered.setdefault(k, []).append(ds)

    mapping = {}
    unresolved = []
    ambiguous = {}

    for ts in sorted(set(target_sessions)):
        candidates = []
        for k in _target_alias_keys(ts):
            hits = key_to_discovered.get(k, [])
            if len(hits) == 1:
                mapping[ts] = hits[0]
                candidates = []
                break
            if len(hits) > 1:
                candidates = hits
        else:
            if candidates:
                ambiguous[ts] = sorted(set(candidates))
            else:
                unresolved.append(ts)

    return mapping, unresolved, ambiguous


def load_targets() -> pd.DataFrame:
    for p in [A_IDS_CSV, B_IDS_CSV]:
        if not p.exists():
            raise FileNotFoundError(f"Missing IDs CSV: {p}")

    a_ids = pd.read_csv(A_IDS_CSV)[["session", "condition", "cluster_id"]].copy()
    b_ids = pd.read_csv(B_IDS_CSV)[["session", "condition", "cluster_id"]].copy()
    a_ids["tuned_label"] = "A_only"
    b_ids["tuned_label"] = "B_only"

    targets = pd.concat([a_ids, b_ids], ignore_index=True)
    targets["cluster_id"] = pd.to_numeric(targets["cluster_id"], errors="coerce")
    targets = targets.dropna(subset=["session", "cluster_id"]).copy()
    targets["cluster_id"] = targets["cluster_id"].astype(int)
    return targets.drop_duplicates().sort_values(["tuned_label", "session", "cluster_id"]).reset_index(drop=True)


def find_session_processed_dirs(root: Path) -> dict:
    if not root.exists():
        raise FileNotFoundError(f"Experimental data root not found: {root}")

    out = {}
    for video_csv in root.rglob("full_video_dataframe.csv"):
        processed_dir = video_csv.parent
        if processed_dir.name != "processed_data":
            continue
        session_name = processed_dir.parent.name
        out.setdefault(session_name, processed_dir)
    return out


def ensure_bool_columns(vdf_raw: pl.DataFrame) -> pl.DataFrame:
    bool_cols = ["shelter", "barrier_present", "barrier_flipped", "EscapePeriod", "OutofshelterIdx", "homingPeriod"]
    out = vdf_raw
    for col in bool_cols:
        if col in out.columns:
            out = out.with_columns(pl.col(col).cast(pl.Boolean))
    return out


def add_bin_edges(df_pd: pd.DataFrame, nbins: int):
    x_min, x_max = np.nanmin(df_pd["x"].values), np.nanmax(df_pd["x"].values)
    y_min, y_max = np.nanmin(df_pd["y"].values), np.nanmax(df_pd["y"].values)
    eps = 1e-9
    x_edges = np.linspace(x_min, x_max + eps, nbins + 1)
    y_edges = np.linspace(y_min, y_max + eps, nbins + 1)
    return x_edges, y_edges


def apply_bins(df_pd: pd.DataFrame, x_edges: np.ndarray, y_edges: np.ndarray):
    nbins_x = len(x_edges) - 1
    nbins_y = len(y_edges) - 1
    df_pd["x_bins"] = np.clip(np.digitize(df_pd["x"].values, x_edges) - 1, 0, nbins_x - 1)
    df_pd["y_bins"] = np.clip(np.digitize(df_pd["y"].values, y_edges) - 1, 0, nbins_y - 1)


def spikes_per_sec_grid(df_cond: pd.DataFrame, df_cond_clu: pd.DataFrame, nbins: int, fps: float) -> np.ndarray:
    occ = df_cond.groupby(["y_bins", "x_bins"]).size().unstack(fill_value=0)
    occ = occ.reindex(index=np.arange(nbins), columns=np.arange(nbins), fill_value=0)

    if df_cond_clu.empty:
        spk = pd.DataFrame(0, index=occ.index, columns=occ.columns)
    else:
        spk = df_cond_clu.groupby(["y_bins", "x_bins"])["spike_count"].sum().unstack(fill_value=0)
        spk = spk.reindex(index=occ.index, columns=occ.columns, fill_value=0)

    with np.errstate(divide="ignore", invalid="ignore"):
        rate = spk.values / np.where(occ.values == 0, np.nan, occ.values) * fps
    return rate


def robust_min_max(arrs, low=2, high=98):
    flat = np.concatenate([a.ravel() for a in arrs if a is not None])
    finite_vals = flat[np.isfinite(flat)]
    if finite_vals.size == 0:
        return 0.0, 1.0
    vmin = np.percentile(finite_vals, low)
    vmax = np.percentile(finite_vals, high)
    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


def load_session_data(processed_dir: Path):
    video_csv = processed_dir / "full_video_dataframe.csv"
    spike_csv = processed_dir / "spike_count_by_frame_and_goodcluster.csv"
    if not video_csv.exists() or not spike_csv.exists():
        return None, None

    vdf_raw = ensure_bool_columns(pl.read_csv(video_csv))
    sdf_raw = pl.read_csv(spike_csv)
    sdf = sdf_raw.rename({"spike_aligned_to_frame": "frame"}).with_columns(pl.col("frame").cast(pl.Int64))
    sdf_pd = sdf.select(["frame", "spike_count", "spike_clusters"]).to_pandas()
    return vdf_raw, sdf_pd


In [ ]:
# -----------------------
# Load and resolve session names
# -----------------------
targets = load_targets()

if SESSION_PREFIX_FILTER:
    prefixes = tuple(str(x).lower() for x in SESSION_PREFIX_FILTER)
    targets = targets[targets["session"].astype(str).str.lower().str.startswith(prefixes)].copy()

session_to_processed = find_session_processed_dirs(EXPERIMENTAL_DATA_ROOT)

resolved_map, unresolved, ambiguous = resolve_session_names(
    target_sessions=targets["session"].tolist(),
    discovered_session_names=list(session_to_processed.keys()),
)

targets = targets.copy()
targets["session_discovered"] = targets["session"].map(resolved_map)
keep = targets.dropna(subset=["session_discovered"]).copy()

print(f"Target rows (after prefix filter): {len(targets):,}")
print(f"Discovered sessions: {len(session_to_processed):,}")
print(f"Resolved target rows: {len(keep):,} / {len(targets):,}")
print(f"Resolved sessions: {keep['session'].nunique():,} / {targets['session'].nunique():,}")
print(f"Unresolved sessions: {len(unresolved):,}")
print(f"Ambiguous sessions: {len(ambiguous):,}")

if unresolved:
    print("First unresolved sessions:")
    print(unresolved[:20])

if ambiguous:
    print("First ambiguous sessions:")
    for i, (k, v) in enumerate(sorted(ambiguous.items())):
        if i >= 10:
            break
        print(f"  {k} -> {v}")

if MAX_PLOTS is not None:
    keep = keep.head(int(MAX_PLOTS)).copy()

keep.head()


In [ ]:
# -----------------------
# Plot + save one figure per tuned cell (memory-light path)
# -----------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

saved = 0
shown_image_preview = 0
skipped_session = 0
skipped_cluster = 0

for session, sess_rows in keep.groupby("session", sort=True):
    session_key = str(sess_rows["session_discovered"].iloc[0])
    processed_dir = session_to_processed.get(session_key)

    if processed_dir is None:
        skipped_session += len(sess_rows)
        print(f"[missing discovered session] {session} -> {session_key}")
        continue

    vdf_raw, sdf_pd = load_session_data(processed_dir)
    if vdf_raw is None or sdf_pd is None:
        skipped_session += len(sess_rows)
        print(f"[missing csvs] {session} -> {processed_dir}")
        continue

    spike_clusters_present = set(pd.to_numeric(sdf_pd["spike_clusters"], errors="coerce").dropna().astype(int).unique())

    # Shared bin edges per session
    vpos = vdf_raw.select(["frames", "mouse_x_position", "mouse_y_position"]).rename(
        {"frames": "frame", "mouse_x_position": "x", "mouse_y_position": "y"}
    )
    vpos_pd = vpos.to_pandas()
    x_edges, y_edges = add_bin_edges(vpos_pd, NBINS)

    # Precompute condition occupancy/base frames only (no spike merge yet)
    cond_base = {}
    for _, cond_key in CONDITIONS:
        vdf_cond_pl = filter_video_dataframe(vdf_raw, cond_key, outofshelter=True, exclude_escape=True)
        base = (
            vdf_cond_pl.select(["frames", "mouse_x_position", "mouse_y_position"])
            .rename({"frames": "frame", "mouse_x_position": "x", "mouse_y_position": "y"})
            .with_columns(pl.col("frame").cast(pl.Int64))
            .to_pandas()
        )
        if base.empty:
            cond_base[cond_key] = base
            continue
        apply_bins(base, x_edges, y_edges)
        cond_base[cond_key] = base[["frame", "x", "y", "x_bins", "y_bins"]]

    print(f"\nSession {session} ({session_key}) | targets={len(sess_rows)}")

    for _, row in sess_rows.iterrows():
        cluster_id = int(row["cluster_id"])
        tuned_label = str(row["tuned_label"])
        source_condition = str(row["condition"])

        if cluster_id not in spike_clusters_present:
            skipped_cluster += 1
            print(f"  [missing cluster] unit {cluster_id} ({tuned_label})")
            continue

        # only this cluster's spikes (smaller merge footprint)
        clu_spk = sdf_pd[sdf_pd["spike_clusters"] == cluster_id][["frame", "spike_count"]].copy()
        if clu_spk.empty:
            skipped_cluster += 1
            print(f"  [no spikes] unit {cluster_id} ({tuned_label})")
            continue

        rates = []
        totals = []

        for _, cond_key in CONDITIONS:
            base = cond_base[cond_key]
            if base.empty:
                rates.append(np.full((NBINS, NBINS), np.nan))
                totals.append(0)
                continue

            # Occupancy from behavior only
            occ = base.groupby(["y_bins", "x_bins"]).size().unstack(fill_value=0)
            occ = occ.reindex(index=np.arange(NBINS), columns=np.arange(NBINS), fill_value=0)

            # Merge this cluster only
            df_unit = base.merge(clu_spk, on="frame", how="left")
            df_unit["spike_count"] = df_unit["spike_count"].fillna(0)
            df_spk = df_unit[df_unit["spike_count"] > 0]
            totals.append(int(df_unit["spike_count"].sum()))

            if df_spk.empty:
                spk = pd.DataFrame(0, index=occ.index, columns=occ.columns)
            else:
                spk = df_spk.groupby(["y_bins", "x_bins"])["spike_count"].sum().unstack(fill_value=0)
                spk = spk.reindex(index=occ.index, columns=occ.columns, fill_value=0)

            with np.errstate(divide="ignore", invalid="ignore"):
                rate = spk.values / np.where(occ.values == 0, np.nan, occ.values) * FPS
            rates.append(rate)

            del df_unit, df_spk, occ, spk

        vmin, vmax = robust_min_max(rates, 2, 98)
        cmap = plt.get_cmap(COLORMAP).copy()
        cmap.set_bad(color="0.8")

        fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.8), sharex=True, sharey=True)
        cbar_ax = fig.add_axes([0.92, 0.20, 0.02, 0.60])
        im_last = None

        for ax, (label, _), rate, tot in zip(axs, CONDITIONS, rates, totals):
            rate_masked = np.ma.array(rate, mask=~np.isfinite(rate))
            im_last = ax.imshow(
                rate_masked,
                origin="lower",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                interpolation="nearest",
                extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                aspect="equal",
            )
            ax.set_title(f"{label.replace('_', ' ')}\nspikes={tot}", fontsize=12)
            ax.set_xticks([])
            ax.set_yticks([])
            for s in ["top", "right", "left", "bottom"]:
                ax.spines[s].set_visible(False)
            ax.invert_yaxis()

        if im_last is not None:
            cbar = fig.colorbar(im_last, cax=cbar_ax)
            cbar.set_label("Spikes/s", rotation=90)

        fig.suptitle(f"{session} | unit {cluster_id} | {tuned_label} | source={source_condition}", fontsize=14)
        fig.subplots_adjust(left=0.03, right=0.90, top=0.86, bottom=0.06, wspace=0.08)

        out_name = f"{session}__unit_{cluster_id}__{tuned_label}__{source_condition}.png"
        out_path = OUT_DIR / out_name
        fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
        plt.close(fig)

        if shown_image_preview < MAX_IMAGE_PREVIEWS:
            display(Image(filename=str(out_path)))
            shown_image_preview += 1

        saved += 1
        print(f"  saved: {out_name}")

        del clu_spk, rates, totals
        gc.collect()

    del cond_base, vdf_raw, sdf_pd, vpos, vpos_pd
    gc.collect()

print("\nDone")
print(f"Saved figures: {saved}")
print(f"Shown image previews: {shown_image_preview}")
print(f"Skipped rows (missing session): {skipped_session}")
print(f"Skipped rows (cluster not found/no spikes): {skipped_cluster}")
print(f"Output dir: {OUT_DIR}")
